# A3 · Corrección telúrica

**Spec:** [`docs/spec_A3_codex_telluric.md`](../docs/spec_A3_codex_telluric.md)  |  **Bloque:** A · Reducción  |  **Run por defecto:** `ROXs12b_realigned`

Corrige absorción telúrica para producir `cube_telcorr.fits`.

| | |
|---|---|
| **Entrada** | Cubo (post-cielo) |
| **Salida (QC/productos)** | `cube_telcorr.fits` |
| **Consume aguas abajo** | A4, B1 |


## Qué es la corrección telúrica y por qué STD_TELLURIC (no molecfit)

La atmósfera terrestre imprime **bandas de absorción** (O₂, H₂O) sobre el espectro, sobre todo en el rojo (>6800 Å): O₂ B ~6870 Å, la fuerte O₂ A ~7600 Å, y H₂O en ~7200/8200/9300 Å. **No son astrofísicas** — para recuperar la forma real del continuo (del compañero, muy rojo) hay que **dividir por la transmisión atmosférica**.

Dos caminos:
- **molecfit** — ajuste de un modelo físico de la atmósfera (lo preferido).
- **STD_TELLURIC** — la transmisión *observada* en la estrella estándar, escalada a la masa de aire de la ciencia por Beer–Lambert: `T_sci(λ) = T_std(λ)^(X_sci/X_std)`.

**Aquí molecfit FALLÓ:** el perfil atmosférico **GDAS** para la fecha/coordenada no estaba disponible → el χ² quedó **congelado** (no mejora entre iteraciones) → transmisión→0, inutilizable. Es un problema de **configuración** (GDAS ausente en el `telluriccorr` instalado), no contaminación estelar. Por eso se usó STD_TELLURIC escalado — el respaldo habitual en MUSE.

**Ventanas protegidas (`T ≡ 1`, la corrección NO se aplica):**
- **6540–6590 Å** = Hα del compañero (diagnóstico de acreción).
- **5780–6050 Å** = láser AO de NFM.

**Consecuencia clave:** Hα está esencialmente libre de telúricas y su ventana está protegida → **A3 NUNCA afecta el resultado científico** (el límite de Hα). A3 solo importa para la fidelidad del **continuo rojo** que usan D1 y el modelado.


## Cómo ejecutar de forma independiente

> ⚠️ **Etapa no re-ejecutable desde raw en este repo.** En la poda WP-10 se borraron los intermedios regenerables (`muse_scibasic`, `muse_scipost`, …). Se conservaron los productos finales y todo el QC. Este notebook **audita** el producto/QC existente y documenta el comando histórico.

Comando histórico (referencia, requiere los raw + `esorex`):

```bash
conda activate MUSE
bash scripts/telluric.sh
```


In [ ]:
import os, sys
# Añade notebooks/ (para _nbcommon) y la RAÍZ del repo (para importar musepipe),
# funcione el cwd en notebooks/ o en la raíz del repo.
_here = os.getcwd()
if os.path.basename(_here) != 'notebooks' and os.path.isdir(os.path.join(_here, 'notebooks')):
    _here = os.path.join(_here, 'notebooks')
for _p in (_here, os.path.dirname(_here)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
_root = str(nb.project_root())
if _root not in sys.path:
    sys.path.insert(0, _root)   # asegura 'import musepipe'
RUN_ID = nb.resolve_run_id(None)
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Auditar

Etapa de solo-auditoría: se carga el producto/QC más abajo.


## Resultados que llevaron a la conclusión

El run realineado **no** emitió `stage00t_qc.json`; reutiliza el mismo método y transmisión. Las métricas de verificación provienen del QC del run crudo (`ROXs12b_raw`), que documenta la reducción telúrica de referencia.


In [ ]:
TELL_REF_RUN = 'ROXs12b_raw'   # QC de referencia: este run no emitió stage00t_qc.json
qt = nb.load_qc('stages/stage00t_qc.json', TELL_REF_RUN)
print(f'QC telúrico mostrado: run {TELL_REF_RUN!r} (run activo del notebook: {RUN_ID!r})')
print()
d, fit, ver = qt['decision'], qt['fit'], qt['verification']
print('Decisión:', d['verdict'], '| aplicado:', d['telluric_applied'],
      '| checkpoint:', d['user_checkpoint'], '| umbral:', d['threshold_pct'], '%')
print('Método:', d['method'])
print()
print(f"Escala airmass: X_std={fit['airmass_std']} -> X_sci={fit['airmass_sci']}"
      f"  ({fit['scaling']} = {fit['airmass_sci']/fit['airmass_std']:.3f})")
print()
print('Profundidad de banda pre -> post:')
pp = ver['v1_o2_depth_pre_post_pct']
print(f'  O2 B (~6870 A): {pp[0]}% -> {pp[1]}%')
print(f"  fuera de bandas sin cambio: {ver['v2_outside_bands_unchanged']}")
print(f"  Halpha intacta: {ver['v3_halpha_untouched']}   transmisión física [0,1]: {ver['v4_transmission_physical']}")
print()
print('molecfit (por qué no):')
print(' ', (qt.get('open_issues') or ['—'])[0])


## De dónde sale la corrección: la transmisión aplicada

**FITS usado:** `TELLURIC_TRANS.fits` (transmisión 1D derivada de `STD_TELLURIC_0001.fits` de `muse_standard`, escalada a la masa de aire de la ciencia). Archivo pequeño — no hace falta el cubo.

El gráfico muestra la transmisión vs λ: se ven las bandas telúricas (O₂ B ~6870, la fuerte O₂ A ~7600, H₂O ~7200/8200/9300) y las **ventanas protegidas** (gris, `T ≡ 1`) — la de Hα (6540–6590) queda plana justo antes de O₂ B.


In [ ]:
MAKE_PLOT = True   # archivo pequeño; requiere kernel MUSE (astropy)
if MAKE_PLOT:
    try:
        import os
        import numpy as np
        import matplotlib.pyplot as plt
        from astropy.io import fits

        qc = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
        cube_path = qc.get('input_cube') or qc.get('cube', {}).get('file')
        trans_path = os.path.join(os.path.dirname(cube_path), 'TELLURIC_TRANS.fits')
        if not os.path.exists(trans_path):
            trans_path = str(nb.project_root() / 'runs' / 'ROXs12b_raw' / 'raw_reduction' / 'TELLURIC_TRANS.fits')
        print('FITS usado:', trans_path)

        h = fits.open(trans_path); t = h[1].data
        wave = np.asarray(t['wave_A'], dtype=float)
        trans = np.asarray(t['transmission'], dtype=float)

        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(wave, trans, lw=1.1, color='tab:blue')
        for i, (a, b) in enumerate([(6540, 6590), (5780, 6050)]):
            ax.axvspan(a, b, color='0.6', alpha=0.35,
                       label='ventana protegida (T≡1)' if i == 0 else None)
        for x, lab in [(6870, 'O₂ B'), (7200, 'H₂O'), (8200, 'H₂O'), (9300, 'H₂O')]:
            ax.annotate(lab, (x, np.interp(x, wave, trans)), textcoords='offset points',
                        xytext=(0, -14), ha='center', fontsize=8, color='tab:red')
        ax.set_xlabel('λ [Å]'); ax.set_ylabel('Transmisión telúrica aplicada')
        ax.set_title('A3 · TELLURIC_TRANS.fits (STD_TELLURIC escalado a airmass sci)')
        ax.set_ylim(0, 1.05); ax.legend(fontsize=8); fig.tight_layout()

        outdir = nb.run_dir(RUN_ID) / 'plots' / 'a3_telluric'
        outdir.mkdir(parents=True, exist_ok=True)
        fig.savefig(outdir / 'transmission.png', dpi=110)
        print('figura ->', outdir / 'transmission.png')
        plt.show(); h.close()
    except Exception as e:
        print('No se pudo generar el plot:', type(e).__name__, e)
        print('Necesita el kernel MUSE (astropy) y TELLURIC_TRANS.fits en disco.')


## Decisiones y notas
- **STD_TELLURIC + escala por airmass**, NO molecfit (no convergió por perfil GDAS ausente). Método de respaldo habitual en MUSE. · [`docs/a3_telluric_justification.md`](../docs/a3_telluric_justification.md)
- Etapa **condicional**: `verdict=needed` por umbral (profundidad máx 6.76% > 3%) + checkpoint humano **aprobado**.


## Conclusión (registrada)

**A3: corrección telúrica APLICADA por STD_TELLURIC escalado a airmass (`telluric_applied = True`, `needed`, checkpoint aprobado).**

- **Fecha:** QC telúrico de referencia 2026-07-06 (`ROXs12b_raw`); justificación para paper 2026-07-10 (`docs/a3_telluric_justification.md`).
- **Datos:** `STD_TELLURIC_0001.fits` (muse_standard), escala airmass 1.087 → 1.158 (exp. 1.065); aplicado → `cube_telcorr.fits`; transmisión en `TELLURIC_TRANS.fits`.
- **Evidencia:** O₂ B 6.76% → 0.6%; continuo fuera de bandas sin cambio; Hα intacta (ventana protegida); STAT escala como T².
- **molecfit no convergió** (perfil GDAS ausente → χ² congelado): problema de configuración documentado; residuo O₂ ~0.6% presupuestado para D2.
- **El realineado no emitió su propio `stage00t_qc.json`** (usa el método/transmisión del crudo). Pendiente: regenerar un QC telúrico propio si se quiere métrica pre/post.
- **Impacto en Hα = nulo** (ventana protegida) → el límite de acreción no depende de esta corrección.
